## Auditoría de Calidad de Datos

In [80]:
#Imports
import pandas as pd

##### Carga de archivos y validación

In [103]:
# Carga de archivos
transactions = pd.read_csv('Datasets/transactions.csv')
transaction_items = pd.read_csv('Datasets/transaction_items.csv')
stores = pd.read_csv('Datasets/stores.csv')
products = pd.read_csv('Datasets/products.csv')
vendors = pd.read_csv('Datasets/vendors.csv')
store_promotions = pd.read_csv('Datasets/store_promotions.csv')

for nombre, df in [('transactions', transactions), ('transaction_items', transaction_items),
                   ('stores', stores), ('products', products),
                   ('vendors', vendors), ('store_promotions', store_promotions)]:
    print("------ Detalle tabla ------ " )
    print(f"{nombre}: {df.shape[0]} filas, {df.shape[1]} columnas")
    print("------ Nulos por campo ------ " )
    print(df.isnull().sum()) 

------ Detalle tabla ------ 
transactions: 174880 filas, 8 columnas
------ Nulos por campo ------ 
transaction_id           0
customer_id         104632
transaction_date         0
store_id                 0
total_amount             0
payment_method           0
loyalty_card             0
status                   0
dtype: int64
------ Detalle tabla ------ 
transaction_items: 542015 filas, 6 columnas
------ Nulos por campo ------ 
transaction_item_id    0
transaction_id         0
item_id                0
quantity               0
unit_price             0
was_on_promo           0
dtype: int64
------ Detalle tabla ------ 
stores: 40 filas, 8 columnas
------ Nulos por campo ------ 
store_id        0
store_name      0
country         0
city            0
format          0
size_sqm        0
opening_date    0
region          0
dtype: int64
------ Detalle tabla ------ 
products: 200 filas, 7 columnas
------ Nulos por campo ------ 
item_id       0
item_name     0
brand         0
vendor_id     0
cat

##### Completitud
¿Qué porcentaje de transacciones no tiene customer_id? ¿Es consistente con loyalty_card = FALSE?

In [82]:
trn_total = len(transactions)
trn_sin_customer = transactions['customer_id'].isna().sum()
trn_pct_sin_customer = round(trn_sin_customer / trn_total * 100, 2)

trn_loyalty_false = (transactions['loyalty_card'] == False).sum()
consistencia = ((transactions['customer_id'].isna()) & (transactions['loyalty_card'] == False)).sum()

print(f'Total transacciones : {trn_total:,}')
print(f'Total transacciones sin customer ID: {trn_pct_sin_customer}%')
print(f'loyalty_card = FALSE : {trn_loyalty_false:,}')
print(f'¿Customer id null es Consistente con loyalty card false? : {trn_sin_customer == consistencia}')


Total transacciones : 174,880
Total transacciones sin customer ID: 59.83%
loyalty_card = FALSE : 104,632
¿Customer id null es Consistente con loyalty card false? : True


**Hallazgo:** El 59.83% de las transacciones (104,632 de 174,880) no tiene customer_id, ese valor consistente con loyalty_card = FALSE, esto tambien indica que una gran cantidad de compras son de clientes no identificlables lo que limita la trazabilidad del comportamiento del cliente. Se recomienda establecer metas para aumentar la cantidad clientes con loyaly card.

**Decisión:** Se segmentarán las transacciones entre clientes identificados (customer_id) y no identificados (loyalty_card = FALSE) para análisis posteriores

##### Consistencia
¿El `total_amount` en `transactions` coincide con la suma de `unit_price × quantity` en `transaction_items`?

In [83]:
# Total en transaction_items
items_total = (
    transaction_items
    .assign(total_item = transaction_items['unit_price'] * transaction_items['quantity'])
    .groupby('transaction_id')['total_item']
    .sum()
    .reset_index()
)

df_check = transactions.merge(items_total, on='transaction_id', how='left')

# Validar inconsistencia puede haber diferencias de 1 centavo por rendonde por lo que se toma en cuenta
df_check['es_consistente'] = abs(df_check['total_amount'] - df_check['total_item']) <= 0.01
df_check['diferencia'] = abs(df_check['total_amount'] - df_check['total_item'])

# 4. métricas
total_trn = len(df_check)
consistentes = df_check['es_consistente'].sum()
inconsistentes = df_check[df_check['diferencia'] > 0.01]
pct_consistencia = round(consistentes / total_trn * 100, 2)

print(f'Total transacciones: {total_trn:,}')
print(f'Transacciones consistentes: {consistentes:,}')
print(f'Transacciones inconsistentes: {(total_trn - consistentes):,}')
print(f'Diferencia máxima: {df_check["diferencia"].max():.4f}')
print(f'Diferencia promedio: {inconsistentes["diferencia"].mean():.4f}')
print(f'% Consistencia: {pct_consistencia}%')
inconsistentes[['transaction_id', 'total_amount', 'total_item', 'es_consistente', 'diferencia']].head()

Total transacciones: 174,880
Transacciones consistentes: 173,135
Transacciones inconsistentes: 1,745
Diferencia máxima: 202.6800
Diferencia promedio: 18.3969
% Consistencia: 99.0%


,transaction_id,total_amount,total_item,es_consistente,diferencia
92,TX_00000093,224.40,235.84,False,11.44
148,TX_00000149,970.40,1055.85,False,85.45
262,TX_00000263,48.99,54.45,False,5.46
367,TX_00000368,30.38,33.04,False,2.66
463,TX_00000464,377.55,390.78,False,13.23


**Hallazgo:** El 1 % del total de transasacciones no coincide con la suma de unit_price * quantity de la tabla transaction_items. Se recomienda revisar las transacciones con diferencias en especial las de mayor impacto (hasta 202.68) para observar que evento pudo afectar las cifras. Buscar otras fuentes que pemitan validar los datos e identificar patrones de ser posible. 

**Desición:** Ya que las incosistencias son de un 1% no se descartaran los datos pero se indentificaran para siguientes análisis. Considerando la exclusion dependiendo del la presicición que se ocupe.



##### Unicidad
 ¿Existen `transaction_id` duplicados?

In [84]:
trn_dupl = transactions['transaction_id'].duplicated().sum()
trn_items_dupl = transaction_items['transaction_item_id'].duplicated().sum()

print(f'transaction_id duplicados: {trn_dupl}')
print(f'transaction_item_id duplicados: {trn_items_dupl}')

transaction_id duplicados: 0
transaction_item_id duplicados: 0


**Hallazgo:** No existen duplicados en tablas principales.

**Decisión:** NA.

##### Validez
¿Hay `total_amount` negativos o cero? ¿Hay `unit_price = 0` con `was_on_promo = FALSE`?

In [89]:
negativos_cero = transactions['total_amount'] <= 0
negativos_cero_count = negativos_cero.sum()
pct = round(negativos_cero_count / trn_total * 100, 2)
print(f'Transacciones <= 0: {negativos_cero_count} ({pct}%)')
transactions.loc[negativos_cero].head()

Transacciones <= 0: 3 (0.0%)


,transaction_id,customer_id,transaction_date,store_id,total_amount,payment_method,loyalty_card,status
36042,TX_00036043,NaN,2024-07-28,TIENDA_005,0.0,DIGITAL,False,COMPLETED
65736,TX_00065737,NaN,2025-05-06,TIENDA_008,0.0,CASH,False,COMPLETED
108160,TX_00108161,NaN,2024-07-16,TIENDA_018,0.0,CASH,False,COMPLETED


In [90]:
precio_cero_sin_promo = (transaction_items['unit_price'] == 0) & (transaction_items['was_on_promo'] == False)
precio_cero_sin_promo_count = precio_cero_sin_promo.sum()
pct = round(precio_cero_sin_promo_count / len(transaction_items) * 100, 2)
print(f'unit_price = 0 y was_on_promo = FALSE: {precio_cero_sin_promo_count} ({pct}%)')

transaction_items.loc[precio_cero_sin_promo].head()

unit_price = 0 y was_on_promo = FALSE: 231 (0.04%)


,transaction_item_id,transaction_id,item_id,quantity,unit_price,was_on_promo
1619,TXI_000001620,TX_00000521,ITEM_089,1,0.0,False
5280,TXI_000005281,TX_00001692,ITEM_089,4,0.0,False
7392,TXI_000007393,TX_00002389,ITEM_089,1,0.0,False
8910,TXI_000008911,TX_00002879,ITEM_089,1,0.0,False
11200,TXI_000011201,TX_00003628,ITEM_089,1,0.0,False


**Hallazgos:**

3 transacciones con total_amount <= 0, puede deberse a errores o devoluciones se puede validar el estado de esas transacciones.

231 ítems con `unit_price = 0`, es raro y puede deberse a algun tipo de evento erroneo como un cambio de precio. 

**Decisiones:**
Las transacciones <= 0 se podrían excluir de los análisis 
Los 231 ítems con precio cero sin promo se pueden marcar como tenerlos con alerta se excluir de calculos segun la necesidad 

##### Integridad referencial
¿Hay `store_id` en `transactions` que no existan en `stores`? ¿`vendor_id` en `products` que no existan en `vendors`?

In [97]:
tx_stores_invalidos = transactions[
    ~transactions['store_id'].isin(stores['store_id'])
]

prod_vendors_invalidos = products[
    ~products['vendor_id'].isin(vendors['vendor_id'])
]

print(f'store_id inválidos en transactions : {len(tx_stores_invalidos)}')
print(f'vendor_id inválidos en products    : {len(prod_vendors_invalidos)}')

if len(prod_vendors_invalidos) > 0:
    print()
    print('Productos con vendor_id inválido:')
    print(prod_vendors_invalidos[['item_id','item_name','vendor_id']].head(5))

store_id inválidos en transactions : 0
vendor_id inválidos en products    : 5

Productos con vendor_id inválido:
      item_id                      item_name vendor_id
44   ITEM_045          Alimentos Producto 45   VND_031
77   ITEM_078   Cuidado Personal Producto 78   VND_031
111  ITEM_112  Cuidado Personal Producto 112   VND_031
155  ITEM_156         Alimentos Producto 156   VND_031
188  ITEM_189         Alimentos Producto 189   VND_031


##### Frescura
¿Hay tiendas con gaps de días consecutivos sin transacciones? ¿Son esperables osospechosos?

In [99]:
gaps_sospechosos = []
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])
for store_id, group in transactions.groupby('store_id'):
    fechas = sorted(group['transaction_date'].dt.date.unique())
    for i in range(1, len(fechas)):
        diff = (fechas[i] - fechas[i-1]).days
        if diff > 3:  # gap mayor a 3 dias es sospechoso
            gaps_sospechosos.append({
                'store_id'        : store_id,
                'fecha_inicio_gap': fechas[i-1],
                'fecha_fin_gap'   : fechas[i],
                'dias_gap'        : diff
            })

gaps_df = pd.DataFrame(gaps_sospechosos)
print(f'Gaps > 3 días consecutivos sin transacciones: {len(gaps_df)}')
if len(gaps_df) > 0:
    print(gaps_df.sort_values('dias_gap', ascending=False).to_string())

Gaps > 3 días consecutivos sin transacciones: 1
     store_id fecha_inicio_gap fecha_fin_gap  dias_gap
0  TIENDA_012       2024-09-09    2024-09-17         8


**Hallazgo:** Solo 1 gap sospechoso detectado: `TIENDA_012` estuvo 8 días sin transacciones (09/09/2024 al 17/09/2024). Podría corresponder a cierre temporal, remodelación o falla en el sistema de reporte.

**Decisión:** Marcar como alerta. En análisis de Comp Sales se revisará si este gap afecta el período de comparación. Se recomienda validar con el equipo operativo.

#### Integridad Temporal
¿Existe alguna tienda con transacciones anteriores a su opening_date?

In [101]:
merged_dates = transactions.merge(
    stores[['store_id', 'opening_date']], on='store_id', how='left'
)

antes_apertura = merged_dates[merged_dates['transaction_date'] < merged_dates['opening_date']]

print(f'Transacciones antes de opening_date: {len(antes_apertura)}')
if len(antes_apertura) > 0:
    resumen = antes_apertura.groupby('store_id').agg(
        transacciones_invalidas=('transaction_id', 'count'),
        fecha_minima=('transaction_date', 'min'),
        opening_date=('opening_date', 'first')
    ).reset_index()
    print(resumen.to_string())

Transacciones antes de opening_date: 50
     store_id  transacciones_invalidas fecha_minima opening_date
0  TIENDA_037                       50   2024-05-15   2024-06-01


**Hallazgo:** 50 transacciones tienen fecha anterior a la `opening_date` de su tienda. Esto es un error de integridad temporal — no debería ser posible vender antes de abrir.

**Decisión:** Excluir estas 50 transacciones de todos los análisis posteriores.

##### A/B Test
¿Hay tiendas asignadas simultáneamente a `CONTROL` y `TREATMENT` en `store_promotions`?

In [102]:
variantes_por_tienda = store_promotions.groupby('store_id')['variant'].nunique()
tiendas_ambos_grupos = variantes_por_tienda[variantes_por_tienda > 1]

print(f'Tiendas en ambos grupos (CONTROL y TREATMENT): {len(tiendas_ambos_grupos)}')
print(f'Tiendas afectadas: {list(tiendas_ambos_grupos.index)}')
print()
print('Detalle:')
print(store_promotions[
    store_promotions['store_id'].isin(tiendas_ambos_grupos.index)
].sort_values(['store_id','variant']).to_string())

Tiendas en ambos grupos (CONTROL y TREATMENT): 2
Tiendas afectadas: ['TIENDA_008', 'TIENDA_037']

Detalle:
      store_id          promo_name    variant  start_date    end_date  promo_type
0   TIENDA_008  Exhibicion_Q3_2024    CONTROL  2024-09-01  2024-10-12  EXHIBICION
40  TIENDA_008  Exhibicion_Q3_2024  TREATMENT  2024-09-01  2024-10-12  EXHIBICION
1   TIENDA_037  Exhibicion_Q3_2024    CONTROL  2024-09-01  2024-10-12  EXHIBICION
41  TIENDA_037  Exhibicion_Q3_2024  TREATMENT  2024-09-01  2024-10-12  EXHIBICION


**Hallazgo:** `TIENDA_008` y `TIENDA_037` están asignadas simultáneamente a `CONTROL` y `TREATMENT`. Esto contamina el experimento, no se puede determinar qué efecto medir para estas tiendas.

**Decisión:** Excluir `TIENDA_008` y `TIENDA_037` del análisis A/B Test del Bloque 3.